# Pandas after NumPy — House Prices (Colab notebook)

This notebook is for students who already know **NumPy arrays** and now need the "new stuff" that **Pandas** adds:

- Named columns (and row indexes)
- Mixed dtypes (numbers + text)
- Easy filtering, grouping, missing values
- Basic string operations with `.str`

We'll use the House Pricing dataset that is commonly available on Colab (`/content/sample_data/california_housing_train.csv`).  
If it's not there, we'll fall back to scikit-learn's California Housing dataset.

---

## How to use this notebook in class

- Read each section, run the code cells.
- In **Exercises**, fill in the `YOUR CODE HERE` parts.
- Optional solutions appear in *collapsed* cells (you can keep them hidden during class).


In [1]:
# Setup: imports and display options
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas version:", pd.__version__)


pandas version: 3.0.1


## 1) Load the dataset

We try:
1. Colab sample CSV: `/content/sample_data/california_housing_train.csv`
2. Fallback: `sklearn.datasets.fetch_california_housing()` (no internet needed)

The goal is to get a DataFrame called `df`.


## If we want to install sklearn at home:

*   uv add numpy
*   uv add pandas
*   uv add scikit-learn




In [2]:
from pathlib import Path

csv_path = Path("/content/sample_data/california_housing_train.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path)
    source = "Colab sample_data CSV"
    print("Using the CSV file from:", csv_path," in the Colab environment.")
else:
    print("CSV file not found at:", csv_path, ' downloading from sklearn...')
    from sklearn.datasets import fetch_california_housing
    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()
    # Rename the target to match the CSV naming
    if "MedHouseVal" in df.columns:
        df = df.rename(columns={"MedHouseVal": "median_house_value"})
    source = "sklearn fetch_california_housing()"

print("Loaded:", source)
print("Shape:", df.shape)
print("Types:")
print(df.dtypes)
df.head()


CSV file not found at: \content\sample_data\california_housing_train.csv  downloading from sklearn...
Loaded: sklearn fetch_california_housing()
Shape: (20640, 9)
Types:
MedInc                float64
HouseAge              float64
AveRooms              float64
AveBedrms             float64
Population            float64
AveOccup              float64
Latitude              float64
Longitude             float64
median_house_value    float64
dtype: object


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,median_house_value
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2) Pandas vs NumPy: what's new?

NumPy gives you fast arrays, but you lose meaning:
- column names
- mixed types
- convenient operations across columns

Pandas keeps the speed-y feel of vectorization, but adds **labels** and **table-like operations**.


In [ ]:
# The "metadata" that NumPy doesn't have:
print("Columns:")
print(df.columns)
print("Columns as list:", list(df.columns))

print("\nData types:")
print(df.dtypes)



Columns:
Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households',
       'median_income', 'median_house_value'],
      dtype='object')
Columns as list: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']

Data types:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
dtype: object


In [ ]:
print("\nA few summary stats:")
df.describe()


A few summary stats:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000
mean,-119.562108,35.625225,28.589353,2643.664412,539.410824,1429.573941,501.221941,3.883578,207300.912353
std,2.005166,2.137340,12.586937,2179.947071,421.499452,1147.852959,384.520841,1.908157,115983.764387
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.790000,33.930000,18.000000,1462.000000,297.000000,790.000000,282.000000,2.566375,119400.000000
50%,-118.490000,34.250000,29.000000,2127.000000,434.000000,1167.000000,409.000000,3.544600,180400.000000
75%,-118.000000,37.720000,37.000000,3151.250000,648.250000,1721.000000,605.250000,4.767000,265000.000000
max,-114.310000,41.950000,52.000000,37937.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


### DataFrame → NumPy (and back)

Sometimes you need to drop down to NumPy.

- `df.to_numpy()` gives a 2D array
- `df["col"].to_numpy()` gives a 1D array


In [ ]:
X_np = df.to_numpy()
y_np = df["median_house_value"].to_numpy() if "median_house_value" in df.columns else None

print("X_np shape:", X_np.shape)
print("y_np shape:", None if y_np is None else y_np.shape)

# Back to DataFrame (if you kept the column names)
df_from_np = pd.DataFrame(X_np, columns=df.columns)
df_from_np.head(2)


X_np shape: (17000, 9)
y_np shape: (17000,)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0


## 3) Selecting columns (the "named NumPy")

Pandas has **3** common ways to select:

1. `df["col"]` → a Series (1D, with an index)
2. `df[["col1","col2"]]` → a DataFrame (2D)
3. `.loc[...]` / `.iloc[...]` for label-based / position-based selection


In [ ]:
# Series vs DataFrame
s = df["median_income"] if "median_income" in df.columns else df[df.columns[0]]
print(type(s))
print("Series shape:", s.shape)
s.head()


<class 'pandas.core.series.Series'>
Series shape: (17000,)


,median_income
0,1.4936
1,1.8200
2,1.6509
3,3.1917
4,1.9250


In [ ]:
df.iloc[2]

,2
longitude,-114.5600
latitude,33.6900
housing_median_age,17.0000
total_rooms,720.0000
total_bedrooms,174.0000
population,333.0000
households,117.0000
median_income,1.6509
median_house_value,85700.0000


In [ ]:
df_smaller = df.loc[5:12:2].copy()
display(df_smaller)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
5,-114.58,33.63,29.0,1387.0,236.0,671.0,239.0,3.3438,74000.0
7,-114.59,34.83,41.0,812.0,168.0,375.0,158.0,1.7083,48500.0
9,-114.60,34.83,46.0,1497.0,309.0,787.0,271.0,2.1908,48100.0
11,-114.60,33.60,21.0,1988.0,483.0,1182.0,437.0,1.6250,62000.0


In [ ]:
try:
  df_smaller.loc[0]
except Exception as e:
  print(e.__class__.__name__)

KeyError


In [ ]:
df_smaller.iloc[0]

,5
longitude,-114.5800
latitude,33.6300
housing_median_age,29.0000
total_rooms,1387.0000
total_bedrooms,236.0000
population,671.0000
households,239.0000
median_income,3.3438
median_house_value,74000.0000


In [ ]:
# Multiple columns -> DataFrame
cols = [c for c in ["median_income", "housing_median_age", "median_house_value"] if c in df.columns]
df[cols].head()


,median_income,housing_median_age,median_house_value
0,1.4936,15.0,66900.0
1,1.8200,19.0,80100.0
2,1.6509,17.0,85700.0
3,3.1917,14.0,73400.0
4,1.9250,20.0,65500.0


### `.iloc` vs `.loc`

- `.iloc[row_index, col_index]` uses **integer positions** (like NumPy indexing)
- `.loc[row_label, col_label]` uses **labels** (row index + column names)


In [ ]:
# iloc: first 3 rows, first 4 columns
df.iloc[:3, :4]


,longitude,latitude,housing_median_age,total_rooms
0,-114.31,34.19,15.0,5612.0
1,-114.47,34.40,19.0,7650.0
2,-114.56,33.69,17.0,720.0


In [ ]:
# loc: first 3 rows by label (our default index is 0..N-1), and specific named columns
some_cols = [c for c in ["median_income", "population", "median_house_value"] if c in df.columns]
df.loc[:2, some_cols]


,median_income,population,median_house_value
0,1.4936,1015.0,66900.0
1,1.8200,1129.0,80100.0
2,1.6509,333.0,85700.0


## Exercise 1 — Column selection

Create:
- `income` as a **Series** of the `median_income` column
- `mini` as a **DataFrame** with `median_income` and `median_house_value`

(If your dataset doesn't have `median_income`, use the first numeric column instead.)


In [ ]:
# YOUR CODE HERE
# income = ...
# mini = ...

# Sanity checks (leave these)
print("income type:", type(income))
print("mini shape:", mini.shape)
mini.head()


NameError: name 'income' is not defined

## 4) Filtering rows (boolean masks)

This is the Pandas version of "NumPy boolean indexing", but with column names:

Example:
```python
df[(df["median_income"] > 3) & (df["housing_median_age"] < 30)]
```

Important: Use parentheses around each condition.


In [ ]:
# A realistic filter: "affordable-ish" and not too old housing
if {"median_income", "housing_median_age", "median_house_value"}.issubset(df.columns):
    filtered = df[(df["median_income"] > 3) & (df["housing_median_age"] < 30)]
    print("filtered shape:", filtered.shape)
    filtered.head()
else:
    print("Some columns missing in this dataset; skipping this demo.")


filtered shape: (5960, 9)


In [ ]:
filtered[0,'median_income'] = 200

/tmp/ipython-input-3322796805.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered[0,'median_income'] = 200


### Common helper methods

- `df["col"].between(a, b)`
- `df["col"].isin([...])`
- `df["col"].notna()` / `isna()`


In [ ]:
if {"median_income", "median_house_value"}.issubset(df.columns):
    mask = df["median_income"].between(2, 4) & df["median_house_value"].notna()
    display(df[mask].head())
else:
    print("Some columns missing in this dataset; skipping this demo.")


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
5,-114.58,33.63,29.0,1387.0,236.0,671.0,239.0,3.3438,74000.0
6,-114.58,33.61,25.0,2907.0,680.0,1841.0,633.0,2.6768,82400.0
8,-114.59,33.61,34.0,4789.0,1175.0,3134.0,1056.0,2.1782,58400.0
9,-114.60,34.83,46.0,1497.0,309.0,787.0,271.0,2.1908,48100.0


## Exercise 2 — Filtering

Find rows where:
- `population` is above the median population
- and `median_income` is above 3

Save the result in `df_hi`.

Tip: compute the population median with `.median()`.


In [ ]:
# YOUR CODE HERE
# pop_med = ...
# df_hi = ...

print("df_hi shape:", df_hi.shape)
df_hi.head()


In [ ]:
# Answer:

pop_med = df['population'].median()
df_hi = df[(df['population'] > pop_med) & (df['median_income'] > 3)]


print("df_hi shape:", df_hi.shape)
display(df_hi)



df_hi shape: (5432, 9)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
13,-114.61,34.83,31.0,2478.0,464.0,1346.0,479.0,3.2120,70400.0
67,-115.54,32.97,41.0,2429.0,454.0,1188.0,430.0,3.0091,70800.0
70,-115.55,32.98,24.0,2565.0,530.0,1447.0,473.0,3.2593,80800.0
82,-115.56,32.78,46.0,2511.0,490.0,1583.0,469.0,3.0603,70800.0
98,-115.58,32.78,5.0,2494.0,414.0,1416.0,421.0,5.7843,110100.0
...,...,...,...,...,...,...,...,...,...
16929,-124.11,40.93,25.0,2392.0,474.0,1298.0,461.0,3.5076,73600.0
16953,-124.15,40.79,37.0,2692.0,488.0,1263.0,486.0,3.0216,86400.0
16955,-124.15,40.76,24.0,2858.0,511.0,1388.0,512.0,3.3750,100600.0
16986,-124.19,40.73,21.0,5694.0,1056.0,2907.0,972.0,3.5363,90100.0


## 5) Creating new columns (vectorized, no loops)

Pandas lets you build features quickly:

- Arithmetic: `df["rooms_per_household"] = df["total_rooms"] / df["households"]`
- Buckets: `pd.cut(...)`
- Conditions: `np.where(...)`


In [ ]:
# Create a few engineered features if the columns exist
if {"total_rooms", "households"}.issubset(df.columns):
    df["rooms_per_household"] = df["total_rooms"] / df["households"]

if {"population", "households"}.issubset(df.columns):
    df["people_per_household"] = df["population"] / df["households"]

df.head()


### Quick tip: avoid division surprises

If a column can have zeros, protect yourself:

```python
df["a"] / df["b"].replace(0, np.nan)
```


## 6) GroupBy: "split → apply → combine"

This is one of the biggest Pandas superpowers.

We'll create a simple **income bucket** and compare average house values.


In [ ]:
if "median_income" in df.columns:
    df["income_bucket"] = pd.cut(
        df["median_income"],
        bins=[-np.inf, 2, 4, 6, np.inf],
        labels=["low", "mid", "high", "very_high"]
    )

    gb = df.groupby("income_bucket", observed=True)["median_house_value"].agg(["count", "mean", "median"])
    gb
else:
    print("median_income missing; skipping groupby demo.")


## Exercise 3 — GroupBy

Create a groupby table `gb2` that shows, for each `income_bucket`:

- mean `people_per_household`
- mean `rooms_per_household`

(If those columns don't exist, create them like in the earlier section.)


In [ ]:
# YOUR CODE HERE
# gb2 = ...

gb2


## 7) Missing values (NaN)

- `isna()` tells you what's missing
- `dropna()` removes rows (careful!)
- `fillna()` replaces missing values

We'll *simulate* missing values so you can see the workflow.


In [ ]:
# Make a tiny copy and introduce missing values (demo)
demo = df.copy()

# Pick a column to corrupt (prefer total_bedrooms if present)
col = "total_bedrooms" if "total_bedrooms" in demo.columns else demo.select_dtypes(include="number").columns[0]

rng = np.random.default_rng(0)
idx = rng.choice(demo.index, size=min(500, len(demo)), replace=False)
demo.loc[idx, col] = np.nan

print("Missing values in demo:")
display(demo[[col]].isna().sum())

# Option A: fill with median
demo_filled = demo.copy()
demo_filled[col] = demo_filled[col].fillna(demo_filled[col].median())

# Option B: drop rows with missing
demo_dropped = demo.dropna(subset=[col])

print("Filled missing:", demo_filled[col].isna().sum())
print("Dropped rows new shape:", demo_dropped.shape)


## 8) Pandas string functions: `.str`

Pandas strings live in object/string columns, and you get vectorized string methods via `.str`.

Our dataset is mostly numeric, so we'll create a **text column** on purpose:
- a "region" label from latitude
- and a "note" column we can clean up

Then we will:
- `.str.lower()`
- `.str.contains()`
- `.str.replace()`
- `.str.split()`
- `.str.extract()` (regex)


In [ ]:
# Create a text column based on latitude (if available), else based on index
df2 = df.copy()

if "latitude" in df2.columns:
    df2["region"] = pd.cut(
        df2["latitude"],
        bins=[-np.inf, 34, 37, np.inf],
        labels=["south", "central", "north"]
    ).astype("string")
else:
    # Fallback: make a fake region from index
    df2["region"] = pd.Series(np.where(df2.index % 3 == 0, "north", np.where(df2.index % 3 == 1, "central", "south")), index=df2.index).astype("string")

# Add a "messy" note column
df2["note"] = ("  Region=" + df2["region"].astype("string") + " ;  ")  # extra spaces & punctuation
df2[["region", "note"]].head()


In [ ]:
# 8.1 Basic cleaning
clean = df2["note"].str.strip().str.lower()
clean.head()


In [ ]:
# 8.2 Contains: find rows mentioning 'north'
mask_north = clean.str.contains("north", na=False)
df2.loc[mask_north, ["region", "note"]].head()


In [ ]:
# 8.3 Replace: turn ';' into nothing and '=' into ':'
fixed = clean.str.replace(";", "", regex=False).str.replace("=", ": ", regex=False)
fixed.head()


In [ ]:
# 8.4 Split into two columns: 'region' and the value
parts = fixed.str.split(": ", n=1, expand=True)
parts.columns = ["key", "value"]
parts.head()


In [ ]:
# 8.5 Extract with regex: pull the region word
extracted = fixed.str.extract(r"region:\s*(\w+)", expand=False)
extracted.value_counts(dropna=False).head()


## Exercise 4 — String functions

Using `df2["note"]`:

1. Create `note_clean` by stripping whitespace and converting to lowercase
2. Create `is_central` which is True when the note contains `"central"`
3. Create `region_extracted` using `.str.extract(...)` that extracts the region word

Print the first 5 rows of these three results side-by-side.


In [ ]:
# YOUR CODE HERE
# note_clean = ...
# is_central = ...
# region_extracted = ...

pd.DataFrame({
    "note": df2["note"].head(5),
    "note_clean": note_clean.head(5),
    "is_central": is_central.head(5),
    "region_extracted": region_extracted.head(5),
})


## 9) Mini capstone: end-to-end summary table

Goal:
- Filter: median_income > 3
- Group by: income_bucket
- Show: count + mean median_house_value + mean rooms_per_household


In [ ]:
if {"median_income", "median_house_value"}.issubset(df.columns):
    tmp = df.copy()

    if {"total_rooms", "households"}.issubset(tmp.columns) and "rooms_per_household" not in tmp.columns:
        tmp["rooms_per_household"] = tmp["total_rooms"] / tmp["households"]

    tmp["income_bucket"] = pd.cut(
        tmp["median_income"],
        bins=[-np.inf, 2, 4, 6, np.inf],
        labels=["low", "mid", "high", "very_high"]
    )

    out = (
        tmp[tmp["median_income"] > 3]
        .groupby("income_bucket", observed=True)
        .agg(
            n=("median_house_value", "size"),
            avg_price=("median_house_value", "mean"),
            avg_rooms_per_household=("rooms_per_household", "mean"),
        )
        .sort_index()
    )
    out
else:
    print("Missing columns for capstone; skipping.")


## Optional (instructor notes): common pitfalls

- `df["col"]` is a Series, not a DataFrame. Shape surprises are normal at first.
- Boolean filters need parentheses: `(A) & (B)`
- `and/or` won't work for Series; use `&` and `|`
- `.loc` is for labels, `.iloc` is for positions
- Strings live in `.str`, datetimes live in `.dt`


<details><summary><b>Solutions (optional, keep collapsed during class)</b></summary>

### Exercise 1
</details>

<details><summary><b>Exercise 2 solution</b></summary></details>

<details><summary><b>Exercise 3 solution</b></summary></details>

<details><summary><b>Exercise 4 solution</b></summary></details>